# 02_1 — Mise en place du tracking MLflow

**Projet 6 OpenClassrooms — Modèle de scoring crédit « Prêt à dépenser »**

**Étape 2 des consignes : « Traquez les expérimentations avec MLflow »**. Résultat attendu : des runs visibles dans l'UI MLflow avec les paramètres testés et les scores obtenus.

**Objectif de ce notebook** : poser l'infrastructure de suivi **avant** de comparer les modèles (notebooks 03_x), puis la tester sur un premier run de référence, un modèle naïf (`DummyClassifier`).

**Méthode** : on suit le screencast du cours `p3_c4_mlflow.py` (`MlflowClient`, expérience avec tags, `mlflow.start_run()`, `log_param`, `log_metrics`, `log_model`, `log_input`).

**Rappel : le vocabulaire MLflow**

| Terme | Ce que c'est | Ici |
|---|---|---|
| *Tracking URI* | où MLflow range les runs (fichiers, base SQL ou serveur) | base SQLite `mlflow.db` à la racine du dépôt |
| *Experiment* | un dossier qui regroupe les runs d'une même question | `scoring_credit` |
| *Run* | une exécution : un modèle entraîné avec un jeu de paramètres | un run par modèle / configuration |
| *Params* | ce qu'on **fixe** avant l'entraînement (hyperparamètres, nb de plis...) | `strategy`, `n_splits`... |
| *Metrics* | ce qu'on **mesure** après (AUC, accuracy, coût métier...) | moyenne et écart-type sur les plis |
| *Tags* | des annotations libres pour retrouver et filtrer les runs | étape, notebook, famille de modèle |
| *Artifacts* | des fichiers rattachés au run (modèle sérialisé, figures, JSON) | modèle, `features_used.json` |

In [1]:
# -------------------------- Imports --------------------------
import os
import sys

sys.path.append("..")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # masque un message d'information de MLflow 3

import json
import warnings

import mlflow
import pandas as pd
from mlflow import MlflowClient
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.model_selection import StratifiedKFold

from src.settings import (
    FEATURES_USED_FILE,
    ID_COLUMN,
    MLFLOW_ARTIFACTS_PATH,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_TRACKING_URI,
    TARGET,
    TRAIN_FILE,
    random_state,
)
from src.modeling_functions import perform_cross_validation

print("MLflow", mlflow.__version__)

/Users/thibaultdautheville/Documents/VSCode/Projet 6/MLOPS-Modele-scoring-financier/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow 3.17.2


## 1. Configuration du tracking

**Pourquoi une base SQLite plutôt que le dossier `mlruns/` par défaut ?** Le stockage par fichiers est déprécié dans MLflow 3, et le **Model Registry** (étape de mise en production) exige une base de données. SQLite ne demande aucun serveur ni installation : c'est un simple fichier, ce qui convient aussi à la VM Medef.

Le screencast utilise un serveur lancé à part (`http://127.0.0.1:8080`). Ici, le notebook écrit **directement** dans la base : pas besoin de laisser tourner un serveur pendant l'entraînement. L'UI sert seulement à consulter les runs (section 5).

`mlflow.db` et `mlartifacts/` sont dans le `.gitignore` : l'historique des runs reste propre à chaque machine.

In [2]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Artefacts    :", MLFLOW_ARTIFACTS_PATH)

2026/10/26 17:04:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/26 17:04:19 INFO mlflow.store.db.utils: Updating database tables


Tracking URI : sqlite:////Users/thibaultdautheville/Documents/VSCode/Projet 6/MLOPS-Modele-scoring-financier/mlflow.db
Artefacts    : /Users/thibaultdautheville/Documents/VSCode/Projet 6/MLOPS-Modele-scoring-financier/mlartifacts


## 2. Création de l'expérience

Les **tags d'expérience** documentent le contexte commun à tous les runs (consigne : « ne pas oublier d'annoter les expériences »). On fixe aussi l'emplacement des artefacts à la racine du dépôt : sinon MLflow les rangerait dans le dossier courant, c'est-à-dire `notebooks/`.

Comme dans le screencast, `create_experiment` échoue si l'expérience existe déjà : on la récupère alors avec `set_experiment`. Le notebook peut ainsi être relancé sans erreur.

In [3]:
with open(FEATURES_USED_FILE, "r") as f:
    features_used = json.load(f)

experiment_tags = {
    "projet": "P6 OpenClassrooms - Prêt à dépenser",
    "donnees": "Home Credit, kernel jsaguiar adapté (notebook 01_2)",
    "revision_de_donnees": "v1",
    "nb_features": str(len(features_used)),
    "cout_metier": "FN = 10 x FP",
    "mlflow.note.content": (
        "Scoring crédit : probabilité de défaut (TARGET = 1). "
        "Comparaison de modèles en StratifiedKFold sur le train (80 %), "
        "le test (20 %) est réservé à l'évaluation finale."
    ),
}

try:
    experiment_id = client.create_experiment(
        MLFLOW_EXPERIMENT_NAME,
        artifact_location=MLFLOW_ARTIFACTS_PATH.as_uri(),
        tags=experiment_tags,
    )
    print("Expérience créée :", MLFLOW_EXPERIMENT_NAME)
except mlflow.exceptions.MlflowException:
    experiment_id = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME).experiment_id
    print("Expérience existante :", MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print("experiment_id :", experiment.experiment_id, "| artefacts :", experiment.artifact_location)

Expérience créée : scoring_credit
experiment_id : 1 | artefacts : file:///Users/thibaultdautheville/Documents/VSCode/Projet%206/MLOPS-Modele-scoring-financier/mlartifacts


## 3. Chargement des données d'entraînement

Seul le **train** (80 %) sert aux expérimentations ; le **test** n'est pas chargé ici, pour ne jamais l'utiliser par mégarde dans les choix de modélisation.

In [4]:
train = pd.read_parquet(TRAIN_FILE)
X = train[features_used]
y = train[TARGET]

print(f"X : {X.shape[0]:,} lignes x {X.shape[1]} features | {X.memory_usage().sum() / 1024**2:.0f} Mo")
print(f"Taux de défaut : {y.mean():.2%}")

X : 246,005 lignes x 659 features | 499 Mo
Taux de défaut : 8.07%


## 4. Premier run : `mlflow.start_run()` sur un modèle de référence

**Pourquoi un `DummyClassifier` ?** C'est la **baseline** : un modèle qui prédit toujours la classe majoritaire (« rembourse »). Tout modèle sérieux devra faire mieux. Ce run sert aussi à vérifier que toute la chaîne MLflow fonctionne avant de lancer des entraînements longs.

Le bloc `with mlflow.start_run(...)` ouvre un run, et le **ferme automatiquement** à la sortie du bloc, même en cas d'erreur. Tout ce qui est loggé à l'intérieur lui est rattaché :
- **tags** et **description** : étape, notebook, famille de modèle ;
- **params** : stratégie du Dummy, validation croisée, `random_state`, nombre de features ;
- **metrics** : AUC, accuracy, rappel et précision, en moyenne et écart-type sur les 5 plis (train et validation) ;
- **artifacts** : le modèle réentraîné sur tout le train et la liste des features ;
- **input** : la description du jeu de données (schéma, nombre de lignes, empreinte), pas les données elles-mêmes.

Le seuil de décision et le coût métier seront ajoutés au notebook 03_1.

In [5]:
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
classification_scoring_metrics = ("roc_auc", "accuracy", "recall", "precision")

# Le Dummy ne prédit jamais de défaut : la précision est indéfinie (0/0), sklearn la met à 0 et prévient.
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

run_tags = {
    "etape": "2 - tracking MLflow",
    "notebook": "02_1_mlflow_setup",
    "famille_modele": "baseline",
}

with mlflow.start_run(
    run_name="dummy_most_frequent",
    tags=run_tags,
    description="Baseline : prédit toujours la classe majoritaire (aucun défaut).",
) as run:
    dummy_classifier = DummyClassifier(strategy="most_frequent")

    scores, scores_dict, dummy_classifier = perform_cross_validation(
        X=X,
        y=y,
        model=dummy_classifier,
        cross_val_type=cv,
        scoring_metrics=classification_scoring_metrics,
    )

    mlflow.log_params(
        {
            "model": "DummyClassifier",
            "strategy": dummy_classifier.strategy,
            "cv": "StratifiedKFold",
            "n_splits": n_splits,
            "random_state": random_state,
            "n_features": X.shape[1],
            "n_train_rows": X.shape[0],
        }
    )
    mlflow.log_metrics(scores_dict)

    mlflow.sklearn.log_model(dummy_classifier, name="model")
    mlflow.log_artifact(FEATURES_USED_FILE)

    dataset_abstraction = mlflow.data.from_pandas(
        train.drop(columns=ID_COLUMN), source=str(TRAIN_FILE.name), targets=TARGET, name="train_v1"
    )
    mlflow.log_input(dataset_abstraction, context="training")

print("run_id :", run.info.run_id)
pd.Series(scores_dict).round(4)

2026/10/26 17:04:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


/Users/thibaultdautheville/Documents/VSCode/Projet 6/MLOPS-Modele-scoring-financier/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


run_id : 4077580e3a064c83b9fcc2744a68bc55


/Users/thibaultdautheville/Documents/VSCode/Projet 6/MLOPS-Modele-scoring-financier/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


average_train_roc_auc      0.5000
train_roc_auc_std          0.0000
average_test_roc_auc       0.5000
test_roc_auc_std           0.0000
average_train_accuracy     0.9193
train_accuracy_std         0.0000
average_test_accuracy      0.9193
test_accuracy_std          0.0000
average_train_recall       0.0000
train_recall_std           0.0000
average_test_recall        0.0000
test_recall_std            0.0000
average_train_precision    0.0000
train_precision_std        0.0000
average_test_precision     0.0000
test_precision_std         0.0000
dtype: float64

**Lecture** : l'accuracy est d'environ **92 %** alors que le modèle ne détecte **aucun** défaut (rappel nul). C'est l'illustration du piège des classes déséquilibrées : l'accuracy ne mesure ici que la part de bons clients. L'**AUC de 0,5** (hasard) et le **rappel** disent la vérité, d'où le recours à l'AUC et au coût métier pour comparer les modèles.

## 5. Vérification : les runs sont-ils enregistrés ?

`mlflow.search_runs` renvoie les runs de l'expérience sous forme de DataFrame, ce qui est pratique pour comparer les modèles directement dans le notebook.

In [6]:
runs = mlflow.search_runs(experiment_names=[MLFLOW_EXPERIMENT_NAME])
cols = ["run_id", "tags.mlflow.runName", "start_time", "metrics.average_test_roc_auc", "metrics.average_test_accuracy", "metrics.average_test_recall"]
runs[cols]

,run_id,tags.mlflow.runName,start_time,metrics.average_test_roc_auc,metrics.average_test_accuracy,metrics.average_test_recall
0,4077580e3a064c83b9fcc2744a68bc55,dummy_most_frequent,2026-10-26 16:04:21.221+00:00,0.5,0.91927,0.0


**Visualiser les runs dans l'UI MLflow**

Dans un terminal, **depuis la racine du dépôt** :

```bash
# Mac (uv)
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db

# VM (environnement .venv activé)
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

puis ouvrir http://127.0.0.1:5000. On y retrouve l'expérience `scoring_credit` et le run `dummy_most_frequent` avec ses paramètres, métriques, tags, le modèle et `features_used.json` (onglet *Artifacts*). `Ctrl + C` dans le terminal arrête l'UI.

Si on relance ce notebook, il crée un **nouveau run** à chaque fois (c'est voulu : chaque exécution est tracée). Les runs de test peuvent être supprimés depuis l'UI.

## 6. Bilan de l'étape 2 (première partie)

- ✅ MLflow installé dans l'environnement virtuel du projet (`uv` sur le Mac, `pip` + `.venv` sur la VM) : pas de conflit de versions.
- ✅ Tracking configuré (base SQLite + dossier d'artefacts), hors GitHub.
- ✅ Expérience `scoring_credit` créée et **annotée** (tags, description).
- ✅ Premier run avec `mlflow.start_run()` : paramètres, métriques, tags, modèle, features et dataset loggés.
- ⏭️ Notebooks 03_x : un run par modèle (LogReg, RF, LightGBM, XGBoost, MLP), coût métier et seuil optimal, `mlflow.autolog()` pour les modèles compatibles, puis **Model Registry** pour versionner le meilleur modèle.